# IHES Cube — Bidirectional Symmetry + Reverse Beam Search

This notebook builds independent direct and reverse beams. Both directions use the controlled model's primary score: the preceding public symmetry/reverse run proved that this ranking independently reaches a replay-valid solution from each side, whereas the direction-aware contrast experiment did not produce a blind join. It first maps every reverse-frontier row into the direct projection and performs the required blind exact full-frontier intersection without a supplied midpoint or midpoint hash. If that exact set intersection is empty, an explicitly labelled blind extension scans every one-move child shell against the opposite complete frontier. Shell children are generated candidates, not claimed top-K states. The known cube-106 path is used only to audit true top-K retention and the optional, separate last-slot protection mode. Protected frontiers are never used for either blind join.


In [ ]:
from pathlib import Path
PUZZLE_ID = 106
MODEL_ID = "1778521793"
BEAM_WIDTH = 1_000_000
FORWARD_DEPTHS = tuple(range(12, 17))
REVERSE_DEPTHS = tuple(range(12, 17))
SYMMETRY_ABSOLUTE_INDEX = None  # None selects the identity frame.
PARENT_CHUNK = 250_000
INFERENCE_BATCH = 8_192
DEVICE = "cuda"
SCORING_MODE = "primary-only"
ALLOW_ONE_MOVE_SHELL = True
SHELL_PARENT_CHUNK = 10_000
RUN_PROTECTED_DIAGNOSTIC = True
KNOWN_PATH_106 = "-r2.-d2.-f2.r1.r1.d0.r2.-d0.-r0.-f0.d0.r0.f1.-d0.f1.r2.r1.-d0.-r2.-f1.-f2.d1.r0.d0"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/cheldieva-l/ihes-dual-model-beam-search'
REPOSITORY_REF = "main"
CHECKOUT = Path("/kaggle/working/ihes-dual-model-beam-search")
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(CHECKOUT)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT), "--no-deps", "-q"], check=True)
sys.path.insert(0, str(CHECKOUT))
print(subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import numpy as np
import torch

from ihes_dual.assets import find_competition_assets
from ihes_dual.model import load_mlp2rb
from ihes_dual.puzzle import IHESPuzzle, load_test_state
from ihes_dual.registry import resolve_model

assets = find_competition_assets("/kaggle/input")
puzzle = IHESPuzzle.from_puzzle_info(assets.puzzle_info)
start = load_test_state(assets.test_csv, PUZZLE_ID)
model_spec = resolve_model("/kaggle/input", MODEL_ID)
if model_spec.model_id != MODEL_ID:
    raise AssertionError("the resolved checkpoint is not tied to the configured model ID")
model = load_mlp2rb(model_spec, DEVICE)
print({
    "puzzle_id": PUZZLE_ID,
    "model_id": model_spec.model_id,
    "checkpoint": model_spec.checkpoint.name,
    "epoch": model_spec.epoch,
    "beam_width": BEAM_WIDTH,
    "device": DEVICE,
    "generator_count": puzzle.generator_count,
})
assert puzzle.generator_count == 18


In [ ]:
from ihes_dual.assets import find_symmetry_file
from ihes_dual.bidirectional import known_path_mapping_report
from ihes_dual.symmetry import load_symmetry_frames

frames = load_symmetry_frames(str(find_symmetry_file("/kaggle/input")), puzzle)
if SYMMETRY_ABSOLUTE_INDEX is None:
    frame_index = next(
        index for index, item in enumerate(frames)
        if np.array_equal(item.rotation, np.arange(puzzle.state_size))
    )
else:
    frame_index = int(SYMMETRY_ABSOLUTE_INDEX)
frame = frames[frame_index]
known_path = puzzle.decode_path(KNOWN_PATH_106)
if PUZZLE_ID != 106:
    known_path = None
else:
    assert len(known_path) == 24
    assert puzzle.verify_solution(start, known_path)
    mapping_report = known_path_mapping_report(puzzle, start, frame, known_path)
    assert all(row["exact_mapping"] for row in mapping_report)
    print("known-path mapping checks:", len(mapping_report))
print("symmetry absolute index:", frame_index)


In [ ]:
from dataclasses import asdict
from ihes_dual.beam import BeamConfig, beam_search
from ihes_dual.bidirectional import blind_join_one_move
from ihes_dual.puzzle import invert_path
from ihes_dual.solve import (
    bidirectional_scorers,
    solve_bidirectional,
    write_run_log,
)
from ihes_dual.submission import build_submission, validate_submission

blind_config = BeamConfig(
    beam_width=BEAM_WIDTH,
    max_depth=max(max(FORWARD_DEPTHS), max(REVERSE_DEPTHS)),
    parent_chunk=PARENT_CHUNK,
    inference_batch=INFERENCE_BATCH,
    device=DEVICE,
    autocast=True,
    prune_immediate_inverse=False,
    diagnostic_protect=False,
)
blind = solve_bidirectional(
    puzzle,
    start,
    model,
    blind_config,
    frame,
    forward_depths=FORWARD_DEPTHS,
    reverse_depths=REVERSE_DEPTHS,
    known_original_path=known_path,
    scoring_mode=SCORING_MODE,
)
exact_intersection_found = blind.joined is not None
if blind.joined is None and ALLOW_ONE_MOVE_SHELL:
    blind.joined = blind_join_one_move(
        puzzle,
        start,
        frame,
        blind.forward,
        blind.reverse,
        forward_depths=FORWARD_DEPTHS,
        reverse_depths=REVERSE_DEPTHS,
        parent_chunk=SHELL_PARENT_CHUNK,
    )
ordinary_reports = {
    "forward": blind.forward.diagnostic_report(),
    "reverse": blind.reverse.diagnostic_report(),
}
for direction, report in ordinary_reports.items():
    print(direction, "ordinary first natural drop:", report["first_natural_drop"])
    for row in report["depths"]:
        print(direction, "ordinary", row)
print("blind exact intersection found:", exact_intersection_found)
print("blind exact-or-one-move join found:", blind.joined is not None)
if blind.joined is not None:
    print("blind join kind:", blind.joined.meeting.join_kind)


In [ ]:
protected_report = None
needs_protected_run = any(
    report["first_natural_drop"] is not None for report in ordinary_reports.values()
)
if RUN_PROTECTED_DIAGNOSTIC and known_path is not None and needs_protected_run:
    protected_config = BeamConfig(**{**asdict(blind_config), "diagnostic_protect": True})
    frame_start = frame.rotate_state(start)
    frame_known = frame.to_frame_path(known_path)
    reverse_start = frame.reverse_start(start)
    reverse_known = invert_path(frame_known, puzzle.inverse_move)
    diagnostic_forward_scorer, diagnostic_reverse_scorer = (
        bidirectional_scorers(model, frame_start, SCORING_MODE)
    )
    diagnostic_forward = beam_search(
        puzzle,
        frame_start,
        diagnostic_forward_scorer,
        protected_config,
        diagnostic_path=frame_known,
    )
    diagnostic_reverse = beam_search(
        puzzle,
        reverse_start,
        diagnostic_reverse_scorer,
        protected_config,
        diagnostic_path=reverse_known,
    )
    protected_report = {
        "forward": diagnostic_forward.diagnostic_report(),
        "reverse": diagnostic_reverse.diagnostic_report(),
    }
    for direction, report in protected_report.items():
        print(direction, "first natural drop:", report["first_natural_drop"])
        for row in report["depths"]:
            print(direction, row)
            assert not row["known_state_protected"] or row["known_state_generated"]
elif RUN_PROTECTED_DIAGNOSTIC and known_path is not None:
    protected_report = {
        "skipped": "ordinary true top-K retained every audited path point"
    }

run_dir = OUTPUT_DIR / "runs" / f"model-{MODEL_ID}" / f"puzzle-{PUZZLE_ID}" / "bidirectional"
run_payload = {
    "mode": "bidirectional-symmetry-reverse",
    "model_id": MODEL_ID,
    "puzzle_id": PUZZLE_ID,
    "symmetry_absolute_index": frame_index,
    "blind_config": asdict(blind_config),
    "scoring": SCORING_MODE,
    "forward_depths": FORWARD_DEPTHS,
    "reverse_depths": REVERSE_DEPTHS,
    "mapped_reverse_frontier_rows": {
        str(depth): len(blind.reverse.frontiers[depth].states)
        for depth in REVERSE_DEPTHS
    },
    "reverse_mapping_formula": "mapped[q] = direct_start[reverse_state[q]]",
    "mapping_applied_before_hashing": True,
    "blind_intersector_used_known_midpoint_or_hash": False,
    "protected_frontiers_used_for_blind_join": False,
    "exact_intersection_found": exact_intersection_found,
    "one_move_shell_enabled": ALLOW_ONE_MOVE_SHELL,
    "one_move_shell_parent_chunk": SHELL_PARENT_CHUNK,
    "shell_children_are_retained_top_k": False,
    "join_kind": (
        None if blind.joined is None else blind.joined.meeting.join_kind
    ),
    "meeting": None if blind.joined is None else asdict(blind.joined.meeting),
    "solution": (
        None if blind.joined is None
        else puzzle.encode_path(blind.joined.original_path)
    ),
    "solution_length": (
        None if blind.joined is None else len(blind.joined.original_path)
    ),
    "replay_valid": (
        False if blind.joined is None
        else puzzle.verify_solution(start, blind.joined.original_path)
    ),
    "known_path_mapping": blind.mapping_report,
    "ordinary_forward_diagnostic": ordinary_reports["forward"],
    "ordinary_reverse_diagnostic": ordinary_reports["reverse"],
    "protected_diagnostic": protected_report,
}
write_run_log(run_dir / "run.json", run_payload)
if blind.joined is None:
    raise RuntimeError("blind exact and one-move complete-frontier joins found no meeting")
if not run_payload["replay_valid"]:
    raise AssertionError("blind joined path failed original-coordinate replay")
print("blind meeting:", blind.joined.meeting)
print("blind solution length:", len(blind.joined.original_path))
print("blind solution:", puzzle.encode_path(blind.joined.original_path))


In [ ]:
submission_path = build_submission(
    assets.sample_submission,
    OUTPUT_DIR / "submission.csv",
    puzzle,
    {PUZZLE_ID: blind.joined.original_path},
)
submission_validation = validate_submission(submission_path, assets.test_csv, puzzle)
run_payload["submission_validation"] = submission_validation
write_run_log(run_dir / "run.json", run_payload)
print("submission:", submission_path)
print("validation:", submission_validation)
